In [2]:
!pip install pandas torch numpy

In [5]:
import pandas as pd

df = pd.read_csv("heart_attack_prediction_dataset.csv")
print(df.head())
print(df.describe())

  Patient ID  Age     Sex  Cholesterol Blood Pressure  Heart Rate  Diabetes  \
0    BMW7812   67    Male          208         158/88          72         0   
1    CZE1114   21    Male          389         165/93          98         1   
2    BNI9906   21  Female          324         174/99          72         1   
3    JLN3497   84    Male          383        163/100          73         1   
4    GFO8847   66    Male          318          91/88          93         1   

   Family History  Smoking  Obesity  ...  Sedentary Hours Per Day  Income  \
0               0        1        0  ...                 6.615001  261404   
1               1        1        1  ...                 4.963459  285768   
2               0        0        0  ...                 9.463426  235282   
3               1        1        0  ...                 7.648981  125640   
4               1        1        1  ...                 1.514821  160555   

         BMI  Triglycerides  Physical Activity Days Per Week  

In [14]:
def retrieve_diastolic_blood_pressure(value):
    return value[value.find('/')+1:]

def retrieve_systolic_blood_pressure(value):
    return value[:value.find('/')]

df['Diastolic Blood Pressure'] = df['Blood Pressure'].map(retrieve_diastolic_blood_pressure)
df['Systolic Blood Pressure'] = df['Blood Pressure'].map(retrieve_systolic_blood_pressure)
print(df['Diastolic Blood Pressure'])
print(df['Systolic Blood Pressure'])

df = df.drop('Blood Pressure', axis=1)

0        88
1        93
2        99
3       100
4        88
       ... 
8758     76
8759    102
8760     75
8761     67
8762     67
Name: Diastolic Blood Pressure, Length: 8763, dtype: object
0       158
1       165
2       174
3       163
4        91
       ... 
8758     94
8759    157
8760    161
8761    119
8762    138
Name: Systolic Blood Pressure, Length: 8763, dtype: object


In [ ]:
pivot = int(df.__len__() * 0.8) 

train, test = df[:pivot][:], df[pivot:][:]
train_x, train_y = train.drop('Heart Attack Risk', axis=1), train['Heart Attack Risk']
test_x, test_y = test.drop('Heart Attack Risk', axis=1), test['Heart Attack Risk']
print(pivot)
print(train_x.info())
print(train_y.info())

print(test_x.info())
print(test_y.info())



7010
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7010 entries, 0 to 7009
Data columns (total 26 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Patient ID                       7010 non-null   object 
 1   Age                              7010 non-null   int64  
 2   Sex                              7010 non-null   object 
 3   Cholesterol                      7010 non-null   int64  
 4   Heart Rate                       7010 non-null   int64  
 5   Diabetes                         7010 non-null   int64  
 6   Family History                   7010 non-null   int64  
 7   Smoking                          7010 non-null   int64  
 8   Obesity                          7010 non-null   int64  
 9   Alcohol Consumption              7010 non-null   int64  
 10  Exercise Hours Per Week          7010 non-null   float64
 11  Diet                             7010 non-null   object 
 12  Previous Heart 

In [29]:
import torch
import torch.nn as nn

class DeepNet(nn.Module):
    def __init__(self):
        super(DeepNet, self).__init__()
        self.ff1 = nn.Linear(26, 26*4)
        self.relu1 = nn.ReLU()
        self.ff2 = nn.Linear(26*4,1)
        

    def forward(self, x):
        x = self.ff1(x)
        x = self.relu1(x)
        x = self.ff2(x)
        return x

model = DeepNet()

criterion = nn.MSELoss()
optimizer = torch.optim.Adamax(model.parameters(), lr=0.001)

model.train()
with torch.no_grad():
    for epoch in range(10):
        optimizer.zero_grad()
        outputs = model(train_x)

        loss = criterion(outputs, train_y)
        loss.backward()
        optimizer.step()

model.eval()

    

TypeError: linear(): argument 'input' (position 1) must be Tensor, not DataFrame